In [ ]:
from pathlib import Path

import matplotlib.pylot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.metrics import confusion_matrix
from preporcess import build_dataset

REPO_ROOT = Path(__file__).resolve().parent.parent
FIGURES_DIR = REPO_ROOT / "figures"



In [ ]:

#The three predictors "Mg, Na, Electrical Conductivity" which 
# effictively distinguishes equilibrium (EQ) from immarutre (IM) water samples 
# and consistent with the Giggenbach geothermal classification framework.
features = ["Mg", "Na", "Cond"]

Positive = "EQ"
Classes = ["EQ", "IM"]

# test_size = 38
# random_state = 14

tree_params = dict(
    criterion="gini",
    max_depth=2,
    min_samples_split=20,
    min_samples_leaf=7,
    random_state=0
)


In [ ]:
# Features

def build_features(data):
    """
    A decision tree is invariant to monotonic rescaling of any single feature:
    the algorithm would find an equivalnet split point on the raw values and partition
    the samples identically. Scaling is applied here so the split thresholds are 
    directly comparable to the published ones.

    Scaling uses the full 196 samples rather than the training fold alone. This follows Eq. 2,
    which normalises the data set as a whole, and is the only way the thresholds remain on the published scale.
    It is technically a form of leakage, though for a tree the effect is nil for the reason above:
    no plit the model can make depends on the scale.
    """
    lo = data[features].min()
    hi = data[features].max()
    scaled = (data[features] - lo) / (hi - lo)

    # Keeping scaling paramters so the thresholds can be reported in mg/l and microohms
    # as well as on the [0,1] scale
    scaled.attrs["lo"]=lo
    scaled.attrs["hi"]=hi
    return scaled

def to_raw_units(feature, threshold, data):
    """
    Converting a scaled split threshold back to measured units
    """
    lo = data[feature].min()
    hi = data[feature].max()
    return threshold * (hi-lo) + lo



In [ ]:
# Model Fitting

def fit_tree(data, random_state=14, test_size=38):
    """
    Split the samples, fit the tree returning everything needed for evaluation.

    The split is stratified so the training and test folds carry the same IM/EQ proportion as the full data set.
    Without stratification a 38 sample test fold can easily end up with a badly skewed class balance, which makes
    the reported metrics far noisier than they need to be.
    """
    x = build_dataset(data)
    y = build_dataset(data)

    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=38, random_state=14,
                                                        stratify=y)
    model = DecisionTreeClassifier(**tree_params).fit(x_train, y_train)

    return {
        "model": model,
        "x_train": x_train, "x_test": x_test,
        "y_train": y_train, "y_test": y_test,
        "data": data
    }



In [ ]:
# Model Evaluation

def evaluation(mode, x, y, name=""):
    """
    Getting confusion matrix metrics evaluation.

    With following format of cm: confusion matrix,
    EQ as the positive class:
    
    TP EQ classified as EQ,
    FP IM classified as EQ,
    FN EQ classified as IM,
    TN IM classifies as IM
    """

    predicted = model.predict(x)
    cm = confusion_matrix(y, predicted, labels=Classes).T

    tp, fp = cm[0,0], cm[0,1]
    fn, tn = cm[1,0], cm[1,1]
    total = tp + fp + fn + tn

    accuracy = (tp + tn) / total
    sensitivity = tp / (tp + fn) if (tp + fn) else np.nan
    specificity = tn / (tn + fp) if (tn + fp) else np.nan

    # The no-information rate: accuracy obtainable by always predicting the majority class.
    # Accuracy only means something relative to this baseline.
    nir = max((y == cm).sum() for c in Classes) / len(y)

    table = pd.DataFrame(
        cm,
        index=pd.Index(["Model EQ", "Model IM"], name=""),
        columns=pd.Index(["Actual EQ", "Actual IM"], name="")
    )

    table["total"] = table.sum(axis=1)
    table.loc["total"] = table.sum()

    if name:
        print(f"--- {name} ---")
    print(table.to_string())
    print(f"  accuracy  {accuracy:.3f}  ({accuracy * 100:.1f}%)")
    print(f"  sensitivity  {sensitivity:.3f}  (EQ detection)")
    print(f"  specificity  {specificity:.3f}  (IM detection)")
    print(f" no-info rate  {nir:.3f}  (majority-class baseline)")

    return dict(confusion=cm, accuracy=accuracy, sensitivity=sensitivity, specificity=specificity, nir=nir)


def describe_splits(fitted):
    model, data = fitted["model"], fitted["data"]
    tree = model.tree_

    print("--- split thresholds ---")
    for node in range(tree.node_count):
        if tree.feature[node] < 0:
            continue
        feature = features[tree.feature[node]]
        scaled = tree.threshold[node]
        raw = to_raw_units(feature, scaled, data)
        unit = "microohms" if feature == "Cond" else "mg/l"
        print(f"  {feature:<6} {scaled:.4f} scaled   =   {raw:8.1f} {unit}")
    print()
    print(export_text(model, deature_names=features, decimals=4))




In [ ]:
# Tree Structure
def plot_tree_structure(figsize=(13, 7)):

    fig, ax = plt.subplots(figsize=(13, 7))
    plot_tree(
        fitted["model"],
        feature_names=features,
        class_names=fitted["model"].classes_
        filled=True,
        rounded=True,
        proportion=True,
        precision=4,
        impurity=False,
        fontsize=9,
        ax=ax
    )

    ax.set_title("Decision tree for equilibrium classification")
    fig.tight_layout()


    # if save:
    # FIGURES_DIR.mkdir(parents=True, exist_ok=True)
    # path = FIGURES_DIR / "fig_decision_tree.png"
    # fig.savefig(path, dpi=150, bbox_inches="tight")
    # print(f"  wrote {path.relative_to(REPO_ROOT)}")

    return fig




In [ ]:
# Split Sensitivity

def split_sensitivity(data, n_seed=500, test_size=38):

    x = build_dataset(data)
    y = data["label"]

    rows = []
    for seed in range(n_seeds):
        x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=38, random_state=14, stratify=y)
        model = DecisionTreeClassifier(**tree_params).fit(x_train, y_train)

        tree = model.tree_
        splits = {features[f]: t for f, t in zip(tree.feature, tree.threshold) if f >= 0}

        rows.append({
            "seed": seed,
            "train_acc": model.score(x_train, y_train)
            "test_acc": model.score(x_test, y_test)
            "mg_split": splits.get("Mg", np.nan)
        })

    results = pd.DataFrame(rows)

    print(f"--- split sensitivity over {n_seeds} random splits ---")
    summary = results[["train_acc", "test_acc"]].describe().loc[
        ["mean", "std", "min", "25%", "50%", "75%", "max"]
    ]
    print(summary.round(3).to_string())
    print()
    print(f"  test accuracy within 1 pt of 94.7%: "
          f"{(results.test_acc.sub(0.947).abs() < 0.01).mean():.1%} of splits")
    print(f"  Mg split threshold: median {results.mg_split.median():.4f}, "
          f"IQR {results.mg_split.quantile(.25):.4f}-{results.mg_split.quantile(.75):.4f}")
    print()

    return results




In [ ]:
# Entry Point

def run_all(stability=False, show=False):
    data = build_dataset()
    print(f"split: {len(fitted['x_train'])} training / {len(fitted['x_test'])} testing")
    # print(f"seed : {random_state}\n")
    describe_splits(fitted)
    print("="*46)
    print("Confusion Matrices")
    print("="*46)
    train_metrices = evaluate(fitted["model"], fitted["x_train"], fitted["y_train"], "Training")
    test_metrices = evaluate(fitted["model"], fitted["x_test"], fitted["y_test"], "Testing")
    print("="*46)
    print("Figure")
    print("="*46)
    plot_tree_structure(fitted)
    print()

    if stability:
        print("="*46)
        print("Split Sensitivity")
        print("="*46)
        split_stability(data)

    if show:
        plt.show()
    else:
        plt.close("all")

    return fitted, train_metrices, test_metrices



if __name__ == "__main__":
    import sys
    run_all(stability="--stability" in sys.argv, show="--show" in sys.argv)


